In [1]:
%%capture
!pip install sacrebleu
!pip install transformers==4.40.2
!pip install sentencepiece
!pip install indic-nlp-library
!pip install torch>=1.10.0
!pip install datasets
!pip install tqdm
!pip install bitsandbytes  # For 4-bit quantization of LLM

In [110]:
import sacrebleu
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import logging as hf_logging
from huggingface_hub import list_repo_refs
from indicnlp.tokenize import indic_tokenize
from datasets import load_dataset
from tqdm import tqdm
import os

import time
import logging
import gc
import copy
import functools

from datasets import load_metric, get_dataset_split_names
from sacrebleu.metrics import CHRF

In [3]:
hf_logging.set_verbosity_error()

In [4]:
# Thanks chatgpt for this helpful method
def setup_notebook_logger():
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)

    if logger.hasHandlers():
        logger.handlers.clear()

    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)

    formatter = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s",
                                  datefmt="%H:%M:%S")
    handler.setFormatter(formatter)

    logger.addHandler(handler)
    return logger

logger = setup_notebook_logger()

# Test
logger.info("Notebook logger is working ✅")

[10:39:13] [INFO] Notebook logger is working ✅


In [5]:
def log_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        logger.info(f"{func.__name__}() took {end - start:.2f} seconds")
        return result
    return wrapper

In [52]:
def safe_add_column(dataset, col_name, values):
    if col_name in dataset.column_names:
        dataset = dataset.remove_columns(col_name)
    return dataset.add_column(col_name, values)

In [6]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'
print(f"using {device} device")

using cuda device


In [7]:
DATASET_REPO = "umairhassan02/coco-captions-subset-59k-translated-v3"

In [8]:
def init_models():
    # NLLB
    nllb_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-3.3B")
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-3.3B")
    nllb_model.config.use_cache= False
    nllb_model.gradient_checkpointing_enable()
    nllb_model = nllb_model.to(device)

    return nllb_tokenizer, nllb_model

In [163]:
def translate_with_nllb(nllb_model, nllb_tokenizer, texts, src_lang="eng_Latn", tgt_lang="urd_Arab", batch_size=20):
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating reference translations"):
        batch_texts = texts[i:i + batch_size]

        # Only move tensors we need
        inputs = nllb_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=256)
        input_ids = inputs["input_ids"].to(device, non_blocking=True)
        attention_mask = inputs["attention_mask"].to(device, non_blocking=True)

        with torch.inference_mode(), torch.cuda.amp.autocast():
            translated_tokens = nllb_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                forced_bos_token_id=nllb_tokenizer.lang_code_to_id[tgt_lang],
                max_new_tokens=128,    # instead of max_length=512
                num_beams=1,
            )

        batch_translations = nllb_tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
        translations.extend(batch_translations)

        # cleanup
        del input_ids, attention_mask, inputs, translated_tokens, batch_translations
        gc.collect()
        torch.cuda.empty_cache()

        # for debugging
        #print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
        #print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

    return translations

In [10]:
def tokenize_urdu(text):
    """
    Tokenize Urdu text using indic-nlp-library.
    Returns a list of tokens.
    """
    return indic_tokenize.trivial_tokenize(text, lang='ur')

In [39]:
def calculate_bleu_chrf(metric, hypotheses, references):
    """Corpus BLEU with sacrebleu."""
    bleu_score = bleu_metric.compute(
        predictions=[h.split() for h in hypotheses],
        references=[[r.split()] for r in references]
    )
    
    # SacreBLEU
    sacre_bleu = sacrebleu.corpus_bleu(hypotheses, [references]).score
    
    # chrF
    chrf = CHRF()
    chrf_score = chrf.corpus_score(hypotheses, [references]).score
    return bleu_score["bleu"], sacre_bleu, chrf_score

In [16]:
ds = load_dataset(DATASET_REPO, revision="1.0.5.0.500")

In [13]:
nllb_tokenizer, nllb_model = init_models()

/opt/conda/lib/python3.12/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/conda/lib/python3.12/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [35]:
bleu_metric = load_metric("bleu")

In [117]:
@log_time
def generate_references_in_batches(dataset, nllb_model, nllb_tokenizer, version, splits):
    transformed_dataset = copy.deepcopy(dataset)
    for split in splits:
        logger.info(f"processing {split} split for {version}")
        """Run translation + QE + refinement on a dataset slice."""
        english_captions = transformed_dataset[split]['caption'] #[dataset[split][i]["caption"] for i in range(start_idx, end_idx)]
        urdu_hypotheses = transformed_dataset[split]['translated_captions'] #[dataset[split][i]["translated_captions"] for i in range(start_idx, end_idx)]
    
        # Generate references
        references = translate_with_nllb(nllb_model, nllb_tokenizer, english_captions, "eng_Latn", "urd_Arab")
        transformed_dataset[split] = transformed_dataset[split].add_column('references', references)
    return transformed_dataset

In [130]:
refs = list_repo_refs(DATASET_REPO, repo_type="dataset")
branches = {b.name: False for b in refs.branches}

In [164]:
for branch in tqdm(branches):
    if branches[branch]:
        logger.info(f"branch {branch} is already processed")
    else:
        data = load_dataset(DATASET_REPO, revision=branch)
        splits = get_dataset_split_names(DATASET_REPO, revision=branch)
        tr_data = generate_references_in_batches(data, nllb_model, nllb_tokenizer, branch, splits)
        tr_data.push_to_hub(DATASET_REPO, revision=branch)
        branches[branch] = True
    gc.collect()
    torch.cuda.empty_cache()

  0% 0/63 [00:00<?, ?it/s][12:44:01] [INFO] branch 1.0.5.58000.58549 is already processed
  2% 1/63 [00:00<00:11,  5.60it/s][12:44:01] [INFO] branch 1.0.5.57000.58000 is already processed
  3% 2/63 [00:00<00:10,  5.64it/s][12:44:03] [INFO] processing train split for 1.0.5.56000.57000

Generating reference translations:   0% 0/251 [00:00<?, ?it/s]/tmp/ipykernel_24382/644845508.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast():

Generating reference translations:   0% 1/251 [00:01<04:30,  1.08s/it]
Generating reference translations:   1% 2/251 [00:01<03:56,  1.05it/s]
Generating reference translations:   1% 3/251 [00:02<03:46,  1.10it/s]
Generating reference translations:   2% 4/251 [00:03<03:56,  1.04it/s]
Generating reference translations:   2% 5/251 [00:08<09:11,  2.24s/it]
Generating reference translations:   2% 6/251 [00:09<07:21,  1.80s/it]
Genera

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/5002 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/51 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
[12:52:06] [WARNING] Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


...

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
[18:30:52] [WARNING] Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/2504 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/26 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
[18:30:53] [WARNING] Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
100% 63/63 [5:46:53<00:00, 330.37s/it]
